## Define Xfer Routes Direction

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [3]:
## Data Preparation and Summary
import pandas as pd
import numpy as np
import geopandas as gpd

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df_blue = pd.read_csv(interim_dir + 'survey_blue_line_xfer_routes_on_off_geocode_manual_add.csv')

In [4]:
midcoast_stns = [
    'UTC Station',
    'Executive Drive Station',
    'UCSD Health La Jolla Station',
    'UCSD Central Campus Station',
    'VA Medical Center Station',
    'Nobel Drive Station',
    'Balboa Avenue Station',
    'Clairemont Drive Station',
    'Tecolote Road Station'
]

In [5]:
df_blue['xfer_on_midcoast'] = ((df_blue['xfer_on_name'].isin(midcoast_stns)) | (df_blue['xfer_off_name'].isin(midcoast_stns))).astype(int)

In [6]:
df_blue.groupby('xfer_on_midcoast')['REWEIGHTED_LINKED'].sum()

xfer_on_midcoast
0    11057.375566
1     2210.566455
Name: REWEIGHTED_LINKED, dtype: float64

In [7]:
station_map = {name: idx + 1 for idx, name in enumerate(midcoast_stns)}
station_map

{'UTC Station': 1,
 'Executive Drive Station': 2,
 'UCSD Health La Jolla Station': 3,
 'UCSD Central Campus Station': 4,
 'VA Medical Center Station': 5,
 'Nobel Drive Station': 6,
 'Balboa Avenue Station': 7,
 'Clairemont Drive Station': 8,
 'Tecolote Road Station': 9}

In [8]:
def map_station(name):
    if pd.isna(name) or name == '':
        return None
    return station_map.get(name, 999) # south of Mid-Coast stations

df_blue['xfer_on_idx'] = df_blue['xfer_on_name'].apply(map_station)
df_blue['xfer_off_idx'] = df_blue['xfer_off_name'].apply(map_station)

In [9]:
def apply_missing_idx_override(row):
    # if pd.isna(row['direction']) and row.get('xfer_on_midcoast') == 1:
    if row.get('xfer_on_midcoast') == 1:
        if pd.isna(row['xfer_on_idx']):
            row['xfer_on_idx'] = 999
        if pd.isna(row['xfer_off_idx']):
            row['xfer_off_idx'] = 999
    return row

# Apply the logic to update the columns in place
df_blue = df_blue.apply(apply_missing_idx_override, axis=1)


In [10]:
def get_direction(on_num, off_num):
    if on_num is None or off_num is None:
        return None
    elif on_num < off_num:
        return 'SB'
    elif on_num > off_num:
        return 'NB'
    else:
        return None

df_blue['direction'] = df_blue.apply(
    lambda row: get_direction(row['xfer_on_idx'], row['xfer_off_idx']),
    axis=1
)

In [11]:
df_blue['xfer_on_name'] = df_blue['xfer_on_name'].fillna('south of Mid-Coast stations')
df_blue['xfer_off_name'] = df_blue['xfer_off_name'].fillna('south of Mid-Coast stations')

In [12]:
df_blue[['ID','REWEIGHTED_LINKED','ROUTE_DIRECTION[Code]','xfer_position','xfer_on_name','xfer_off_name','xfer_on_midcoast','direction']].to_csv(output_dir + '2023OBS_blue_line_xfer_dataset.csv',index=False)